## La machine à états (quête 2.5)

`start_orchestrator()` : une boucle `while True` et quatre états.

| État | timeout donné à `listen()` | Ce qu'on attend |
|---|---|---|
| VEILLE | `None` | le mot-clé « Jarvis », indéfiniment |
| ÉCOUTE | `8` (`CONVERSATION_TIMEOUT`) | une réplique, 8 s maximum |
| RÉFLEXION | — | `chat()` chez Ollama |
| PAROLE | — | `speak()` (pyttsx3) |

Trajet normal : `VEILLE → ÉCOUTE → RÉFLEXION → PAROLE → ÉCOUTE → …`
Sortie de conversation : `ÉCOUTE → VEILLE` dès que `transcribe()` rend `None`.

**Le micro ne s'éteint jamais.** Le `with sr.Microphone() as source:` de `main.py` ouvre le flux une seule fois. Seule la *consigne* change à chaque itération : `listen(timeout=None)` en VEILLE, `listen(timeout=8)` en ÉCOUTE. Le « rallumage » n'est donc qu'un nouveau tour de boucle, pas une remise sous tension.

**`timeout` = patience, pas durée d'enregistrement.** `listen(timeout=8)` lève `WaitTimeoutError` si *personne ne commence à parler* dans les 8 s. Une fois la parole commencée, la montre n'a plus d'importance : `phrase_time_limit=None` laisse la phrase finir. En pratique, le micro capte souvent du bruit (écho d'espeak) : Vosk transcrit alors une chaîne vide, et c'est la garde-fou `if not data or not data.strip(): return None` qui ramène en VEILLE — pas le timeout. Deux portes, même destination.

```text
while True:
    data = await asyncio.to_thread(transcribe, r, source, time_out)  # ← la boucle s'arrête ici
    if data is None:  # en ÉCOUTE → VEILLE
        continue
```

## `to_thread()`, event loop et durée de vie du thread

Les appels bloquants (`transcribe()`, `chat()`, `speak()`) partent dans `asyncio.to_thread(...)`. Trois étages à ne pas confondre :

| Étage | Pendant `listen()` |
|---|---|
| Event loop | libre : il peut faire tourner d'autres coroutines |
| Coroutine `start_orchestrator` | **suspendue** sur l'`await` : la boucle `while` ne fait aucune itération de plus |
| Thread du pool | **bloqué** dans `listen()` : c'est lui qui attend vraiment |
| Micro `source` | ouvert, dans le thread principal |

**« ne pas bloquer le programme » ≠ « la boucle continue de tourner ».** La boucle est polie : elle s'efface et reprend seulement quand le thread lui rend son résultat.

**Durée de vie du thread :** pas 8 s, et pas un thread neuf à chaque appel. `to_thread()` emprunte un ouvrier au pool par défaut ; il reste occupé le temps de `transcribe()` (8 s max en ÉCOUTE, sans limite en VEILLE), puis retourne au pool et est réutilisé au tour suivant. Comme les appels sont séquentiels, c'est souvent toujours le même (`ThreadPoolExecutor-0_0`). Il ne disparaît qu'à la fin de `asyncio.run()`, qui éteint le pool.

⚠️ Un thread bloqué ne s'annule pas : pour un bouton « stop » ou un réveil simultané, il faudra repenser la structure. Quête future, pas celle-ci.

## Statistiques

In [ ]:
import httpx
from ai_assistant.config import OLLAMA_BASE_URL, OLLAMA_MODEL, SYSTEM_PROMPT, TIME_OUT

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Réponds en trois phrases courtes : que fais-tu ?"},
]
r = httpx.post(
    f"{OLLAMA_BASE_URL}/api/chat",
    json={"model": OLLAMA_MODEL, "messages": messages, "stream": False},
    timeout=TIME_OUT,
)
r.raise_for_status()
data = r.json()
count = data["eval_count"]
seconds = data["eval_duration"] / 1e9
print("Réponse :", data["message"]["content"])
print("Tokens générés :", count)
print("Durée génération :", round(seconds, 2), "s")
print("Tokens/s :", round(count / seconds, 1))